# Silver Weather

## 1. Business Purpose
Weather Forecast Silver dùng để cung cấp dữ liệu dự báo
thời tiết theo giờ tại từng warehouse của FastOrder.

Dữ liệu này có thể phục vụ:
- phân tích ảnh hưởng thời tiết tới orders / delivery
- logistics planning
- warehouse operations
- các mô hình analytics / forecasting sau này

## 2. Overview Data

In [0]:
import os

account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

### Path

In [0]:
# abfss://<container>@<storage-account>.dfs.core.windows.net/<path>
# /bronze/weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_HCM/ingestion_id=9ece2b40-aacc-57e8-bc3f-cb6b85e8d233/metadata.json

path_metadata = "weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_HCM/ingestion_id=9ece2b40-aacc-57e8-bc3f-cb6b85e8d233/metadata.json"
path_response = "weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_HCM/ingestion_id=9ece2b40-aacc-57e8-bc3f-cb6b85e8d233/response.json"

### Metadata

In [0]:
df_metadata_weather = spark.read.format("json").option("multiline", True).load(f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{path_metadata}")

df_metadata_weather.limit(5).display()

In [0]:
df_metadata_weather.printSchema()

### response

In [0]:
from pyspark.sql import functions as F
df_response_weather = spark.read.format("json").option("multiline", True).load(f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{path_response}")

df_response_weather = (
    df_response_weather
    .withColumn("_source_file_path", F.col("_metadata.file_path"))
    .withColumn("ingestion_date", F.regexp_extract("_source_file_path", r"ingestion_date=([^/]+)", 1))
    .withColumn("warehouse_id", F.regexp_extract("_source_file_path", r"warehouse_id=([^/]+)", 1))
    .withColumn("ingestion_id", F.regexp_extract("_source_file_path", r"ingestion_id=([^/]+)", 1))
)

df_response_weather.limit(5).display()

In [0]:
df_weather = df_response_weather.withColumn(
    "hourly",
    F.arrays_zip(
        "hourly.time",
        "hourly.temperature_2m",
        "hourly.relative_humidity_2m",
        "hourly.precipitation",
        "hourly.wind_speed_10m",
        "hourly.weather_code",
    )
)

df_weather.display()

In [0]:
df_weather = df_weather.withColumn("hourly", F.explode("hourly"))

In [0]:
df_weather = df_weather.select(
    "latitude",
    "longitude",
    "timezone",
    "ingestion_id",
    "warehouse_id",
    "ingestion_date",
    F.col("hourly.time").alias("time"),
    F.col("hourly.temperature_2m").alias("temperature_2m"),
    F.col("hourly.relative_humidity_2m").alias("relative_humidity_2m"),
    F.col("hourly.precipitation").alias("precipitation"),
    F.col("hourly.wind_speed_10m").alias("wind_speed_10m"),
    F.col("hourly.weather_code").alias("weather_code")
    )

df_weather.display()

In [0]:
from pyspark.sql.functions import col
m = df_metadata_weather.alias("m")
r = df_weather.alias("r")
df_weather_joined = r.join(m, col("r.ingestion_id") == col("m.ingestion_id"), how = "inner")

df_weather_joined.display()

## 3. Silver Design
- Grain 
- Key 
- Columns 
- Lineage 
- DQ rules 

### Schema Design

In [0]:
df_weather_forecast_hourly = df_weather_joined.select(
    F.col("m.warehouse_id").alias("warehouse_id"),
    F.col("m.ingestion_id").alias("ingestion_id"),
    F.col("logical_at").alias("snapshot_at"),
    F.col("requested_at").alias("retrieved_at"),
    F.col("time").alias("forecast_time"),
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "weather_code",
    "requested_latitude",
    "requested_longitude",
    "response_latitude",
    "response_longitude"
)

In [0]:
from pyspark.sql.types import TimestampType
df_weather_forecast_hourly = df_weather_forecast_hourly.withColumn(
    "retrieved_at", F.col("retrieved_at").cast(TimestampType())
).withColumn(
    "snapshot_at", F.col("snapshot_at").cast(TimestampType())
).withColumn(
    "forecast_time", F.col("forecast_time").cast(TimestampType())
)


In [0]:
df_weather_forecast_hourly.printSchema()

In [0]:
df_weather_forecast_hourly.limit(5).display()

In [0]:
spark.conf.get("spark.sql.session.timeZone")

In [0]:
df_weather_forecast_hourly = df_weather_forecast_hourly.withColumn(
    "forecast_time",
    F.to_utc_timestamp(
        F.col("forecast_time"),
        "Asia/Ho_Chi_Minh"
    )
)

## Data Quality Profiling

### Null values check

NULL của 3 key columns

In [0]:
df_weather_forecast_hourly.select(
    F.sum(F.col("ingestion_id").isNull().cast("int")).alias("ingestion_id_null_count"),
    F.sum(F.col("warehouse_id").isNull().cast("int")).alias("warehouse_id_null_count"),
    F.sum(F.col("forecast_time").isNull().cast("int")).alias("forecast_time_null_count")
).display()

NULL của 5 weather columns

In [0]:
df_weather_forecast_hourly.select(
    F.sum(F.col("temperature_2m").isNull().cast("int")).alias("temperature_2m_null_count"),
    F.sum(F.col("relative_humidity_2m").isNull().cast("int")).alias("relative_humidity_2m_null_count"),
    F.sum(F.col("precipitation").isNull().cast("int")).alias("precipitation_null_count"),
    F.sum(F.col("wind_speed_10m").isNull().cast("int")).alias("wind_speed_10m_null_count"),
    F.sum(F.col("weather_code").isNull().cast("int")).alias("weather_code_null_count")
).display()

### Outlier values check

humidity ngoài 0–100

In [0]:
df_weather_forecast_hourly.select(
    F.sum(F.when(F.col("relative_humidity_2m") < 0, 1).otherwise(0)).alias("relative_humidity_2m_negative_count"),
    F.sum(F.when(F.col("relative_humidity_2m") > 100, 1).otherwise(0)).alias("relative_humidity_2m_above_100_count")
).display()

precipitation < 0

In [0]:
df_weather_forecast_hourly.select(
    F.sum(F.when(F.col("precipitation") < 0, 1).otherwise(0)).alias("precipitation_negative_count")
).display()

wind_speed < 0

In [0]:
df_weather_forecast_hourly.select(
    F.sum(F.when(F.col("wind_speed_10m") < 0, 1).otherwise(0)).alias("wind_speed_10m_negative_count")
).display()

duplicate theo grain

In [0]:
(
    df_weather_forecast_hourly
    .groupBy(
        "warehouse_id",
        "ingestion_id",
        "forecast_time"
    )
    .agg(
        F.count("*").alias("row_count")
    )
    .filter(
        F.col("row_count") > 1
    )
).display()

## 4. Transformation

## 5. Data Quality


## 6. Validation


## 7. Silver Write